In [3]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 30)#df how much it can show to me
pd.set_option("display.width", 200)

In [4]:
# . LOAD
df = pd.read_csv(r"c:\Users\ASUS\Downloads\Swiggy_Python_Messy_500.csv")
print("Loaded shape:", df.shape)


Loaded shape: (512, 26)


In [5]:
# 1. STRIP WHITESPACE from every text column

str_cols = df.select_dtypes(include=["object"]).columns
for col in str_cols:
    df[col] = df[col].astype(str).str.strip()
    # astype(str) turns real NaN into the text "nan" -> put real NaN back
    df[col] = df[col].replace("nan", np.nan)


In [6]:
# 2. STANDARDIZE MISSING-VALUE TOKENS -> np.nan
#    ("NA", "N/A", "null", "-", "", "unknown", case-insensitive)

missing_tokens = {"na", "n/a", "null", "-", "", "unknown", "not available", "0000-00-00"}
for col in str_cols:
    df[col] = df[col].apply(
        lambda x: np.nan if isinstance(x, str) and x.strip().lower() in missing_tokens else x
    )

# isinstance() = “Is this thing is this type?”
#"Is this value a missing-value word? If yes, replace it with np.nan. Otherwise, keep it unchanged."

In [7]:
# 3. GENDER: collapse M/m/male/MALE -> "Male", F/f/female/FEMALE -> "Female"  

gender_map = {
    "m": "Male", "male": "Male",
    "f": "Female", "female": "Female",
}
df["Gender"] = df["Gender"].apply(
    lambda x: gender_map.get(x.lower(), x) if isinstance(x, str) else x
)
#.get() means “look for this key and give me its value.”

In [8]:
# 4. CITY: fix case + known alias, then Title Case

city_alias = {"bengaluru": "Bangalore"}
def clean_city(c):
    if not isinstance(c, str):
        return c
    key = c.strip().lower()
    key = city_alias.get(key, key)
    return key.title()
df["City"] = df["City"].apply(clean_city)

In [9]:
# 5. PAYMENT MODE: normalize spacing/case to a fixed canonical set

def clean_payment_mode(p):
    if not isinstance(p, str):          #if p is NaN, don't try to clean it
        return p
    key = p.strip().lower().replace(" ", "")
    mapping = {
        "upi": "UPI",
        "creditcard": "Credit Card",
        "debitcard": "Debit Card",
        "cashondelivery": "Cash on Delivery",
        "wallet": "Wallet",
    }
    return mapping.get(key, p.title())
df["PaymentMode"] = df["PaymentMode"].apply(clean_payment_mode)  


In [10]:
# 6. ORDER STATUS: Title Case ("DELIVERED"/"delivered" -> "Delivered")

df["OrderStatus"] = df["OrderStatus"].apply(lambda x: x.title() if isinstance(x, str) else x)

In [11]:
# 7. CUISINE: fix known typos

cuisine_fix = {
    "chineese": "Chinese", "chinese": "Chinese",
    "itallian": "Italian", "italian": "Italian",
    "north-indian": "North Indian", "north indian": "North Indian",
}
df["Cuisine"] = df["Cuisine"].apply(
    lambda x: cuisine_fix.get(x.lower(), x) if isinstance(x, str) else x
)

In [12]:
# 8. RESTAURANT NAME / CUSTOMER NAME: fix stray CAPS + double spaces
df["RestaurantName"] = df["RestaurantName"].str.replace(r"\s+", " ", regex=True)
df["CustomerName"] = df["CustomerName"].apply(
    lambda x: x.title() if isinstance(x, str) else x
)

r means the regular expression (regex) understand:

\s → whitespace (space, tab, etc.)     +  → one or more

" "       → replace them with one space

regex=True → tell Pandas that \s+ is a pattern

In [13]:
# 9. ORDER DATE: parse multiple mixed formats into one datetime dtype
#    (mixed-format columns are the #1 real-world date headache)

df["OrderDate"] = pd.to_datetime(df["OrderDate"], errors="coerce", format="mixed", dayfirst=False)
print("\nUnparseable dates after coercion:", df["OrderDate"].isna().sum())



Unparseable dates after coercion: 7


format="mixed" -->  The column may contain different date formats.

errors="coerce" ---> abc, wrongdate    

 instead of crashing, pandas converts it to: NaT 

 (NaT) -means:Not a Time / missing datetime.

 dayfirst=False
 
Means the parser does not assume the first number is always the day.

In [14]:
# 10. UNITPRICE / TOTALAMOUNT: strip currency symbols/text -> float

def to_number(x):
    if pd.isna(x):
        return np.nan
    if isinstance(x, (int, float)):
        return float(x)
    cleaned = (
        str(x).replace("Rs.", "").replace("Rs", "")
        .replace("₹", "").replace(",", "").strip()
    )
    try:
        return float(cleaned)
    except ValueError:
        return np.nan

df["UnitPrice"] = df["UnitPrice"].apply(to_number)
df["TotalAmount"] = df["TotalAmount"].apply(to_number)

In [15]:
# 11. QUANTITY: fix sign errors, flag zero-quantity orders
df["Quantity"] = df["Quantity"].abs()
zero_qty = (df["Quantity"] == 0).sum()
df.loc[df["Quantity"] == 0, "Quantity"] = np.nan   # can't have a 0-item order
print("Zero-quantity rows set to NaN:", zero_qty)

Zero-quantity rows set to NaN: 6


In [ ]:
# 12. RECONCILE TotalAmount vs Quantity * UnitPrice
#  Find the rows where TotalAmount is wrong.
#  Replace those wrong values with Quantity × UnitPrice (the correct calculated total).

expected_total = (df["Quantity"] * df["UnitPrice"]).round(2)
mismatch_mask = (df["TotalAmount"] - expected_total).abs() > 1.0
print("TotalAmount mismatches corrected:", mismatch_mask.sum())
df.loc[mismatch_mask, "TotalAmount"] = expected_total[mismatch_mask]

TotalAmount mismatches corrected: 18


In [17]:
# 13. AGE: keep only plausible customer ages (10-90), else NaN
# ~ (This symbol means NOT)Find ages that are NOT between 10 and 90.

invalid_age = ~df["Age"].between(10, 90)
print("Invalid ages set to NaN:", invalid_age.sum())
df.loc[invalid_age, "Age"] = np.nan


Invalid ages set to NaN: 37


In [18]:
# 14. RATINGS: RestaurantRating / FoodRating / DeliveryRating must be 1-5

for col in ["RestaurantRating", "FoodRating", "DeliveryRating"]:
    bad = ~df[col].between(1, 5) & df[col].notna()
    print(f"Invalid {col} set to NaN:", bad.sum())
    df.loc[bad, col] = np.nan

Invalid RestaurantRating set to NaN: 10
Invalid FoodRating set to NaN: 7
Invalid DeliveryRating set to NaN: 7


In [19]:
# 15. DISTANCE / DELIVERY TIME: clip obviously impossible values

bad_dist = ~df["DistanceKM"].between(0.1, 40) & df["DistanceKM"].notna()
print("Invalid DistanceKM set to NaN:", bad_dist.sum())
df.loc[bad_dist, "DistanceKM"] = np.nan

bad_time = ~df["DeliveryTimeMin"].between(5, 180) & df["DeliveryTimeMin"].notna()
print("Invalid DeliveryTimeMin set to NaN:", bad_time.sum())
df.loc[bad_time, "DeliveryTimeMin"] = np.nan

Invalid DistanceKM set to NaN: 7
Invalid DeliveryTimeMin set to NaN: 7


In [20]:
# 16. IDS: final whitespace cleanup (already stripped in step 1,
#     kept here as an explicit checkpoint in case new cols are added)
for col in ["OrderID", "CustomerID", "RestaurantID"]:
    df[col] = df[col].str.strip()

In [21]:
# 17. DROP DUPLICATES
#     a) exact duplicate rows
#     b) duplicate OrderID (keep the first occurrence)

before = len(df)
df = df.drop_duplicates()
print("\nExact duplicate rows dropped:", before - len(df))

before = len(df)
df = df.drop_duplicates(subset="OrderID", keep="first")
print("Duplicate OrderIDs dropped:", before - len(df))


Exact duplicate rows dropped: 8
Duplicate OrderIDs dropped: 4


In [ ]:
# 18. FINAL CHECKS
#  index=False   -->   Without it, pandas might save the DataFrame index as an extra column.

df = df.sort_values("OrderID").reset_index(drop=True)
print("\nFinal cleaned shape:", df.shape)
print("\nRemaining missing values per column:")
print(df.isna().sum())

df.to_csv("Swiggy_Python_Cleaned_500.csv", index=False)
print("\nSaved -> Swiggy_Python_Cleaned_500.csv")


Final cleaned shape: (500, 26)

Remaining missing values per column:
OrderID               0
OrderDate             7
CustomerID            0
CustomerName         22
Gender               26
Age                  35
City                 17
Area                 15
RestaurantID          0
RestaurantName        0
Cuisine               0
RestaurantRating     30
ItemName              0
CategoryName          0
Quantity              5
UnitPrice            10
TotalAmount          10
OrderStatus           0
PaymentMode          22
PaymentStatus         0
PartnerName          10
VehicleType           0
DistanceKM           22
DeliveryTimeMin      21
FoodRating          191
DeliveryRating      193
dtype: int64

Saved -> Swiggy_Python_Cleaned_500.csv


In [23]:
sdf = pd.read_csv("Swiggy_Python_Cleaned_500.csv")

In [24]:
sdf.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 26 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   OrderID           500 non-null    object 
 1   OrderDate         493 non-null    object 
 2   CustomerID        500 non-null    object 
 3   CustomerName      478 non-null    object 
 4   Gender            474 non-null    object 
 5   Age               465 non-null    float64
 6   City              483 non-null    object 
 7   Area              485 non-null    object 
 8   RestaurantID      500 non-null    object 
 9   RestaurantName    500 non-null    object 
 10  Cuisine           500 non-null    object 
 11  RestaurantRating  470 non-null    float64
 12  ItemName          500 non-null    object 
 13  CategoryName      500 non-null    object 
 14  Quantity          495 non-null    float64
 15  UnitPrice         490 non-null    float64
 16  TotalAmount       490 non-null    float64
 1